In [ ]:
from __future__ import annotations
import glob, json, os, subprocess, sys
from pathlib import Path
os.environ.setdefault('JAX_COMPILATION_CACHE_DIR', '/kaggle/working/jax-cache')
wheel = [p for p in glob.glob('/kaggle/input/**/*.whl', recursive=True) if Path(p).name.startswith('hghost_jax-')]
if len(wheel) != 1: raise RuntimeError(f'Expected one hghost-jax wheel, found: {wheel}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--quiet', wheel[0]], check=True)
import jax
import numpy as np
from h1jax.config import FalconH1Config
from h1jax.model import parameter_count_for_config
from h1jax.train import build_parser, run
hardware = {'jax': jax.__version__, 'backend': jax.default_backend(), 'device_count': jax.device_count(), 'devices': [str(d) for d in jax.devices()]}
print(json.dumps({'event': 'hardware', **hardware}), flush=True)
if jax.default_backend() != 'tpu' or jax.device_count() != 8: raise RuntimeError('This gate requires the complete TPU v5e-8 slice')
working = Path('/kaggle/working/smoke-input'); working.mkdir(parents=True, exist_ok=True)
cfg = FalconH1Config(); parameters = parameter_count_for_config(cfg)
if parameters != 91_131_072: raise RuntimeError(f'Unexpected parameters: {parameters}')
cfg.to_json(working / 'config.json')
sequence_length, per_device_batch, steps = 512, 4, 2
total_tokens = jax.device_count() * per_device_batch * sequence_length * steps
rng = np.random.default_rng(20260901)
rng.integers(0, cfg.vocab_size, size=total_tokens * 2, dtype=np.uint16).tofile(working / 'train.bin')
args = build_parser().parse_args(['--config', str(working/'config.json'), '--train-bin', str(working/'train.bin'), '--random-init', '--output', '/kaggle/working/h1jax-smoke-output', '--sequence-length', str(sequence_length), '--per-device-batch', str(per_device_batch), '--accumulation-steps', '1', '--total-tokens', str(total_tokens), '--warmup-tokens', str(total_tokens//2), '--learning-rate', '0.001', '--dtype', 'bfloat16', '--save-tokens', '', '--no-save-final-checkpoint', '--log-steps', '1'])
run(args)
completion = json.loads(Path('/kaggle/working/h1jax-smoke-output/training-complete.json').read_text())
if completion.get('completed') is not True or int(completion.get('steps', -1)) != steps: raise RuntimeError(f'Incomplete smoke: {completion}')
report = {'ok': True, **hardware, 'parameters': parameters, 'sequence_length': sequence_length, 'per_device_batch': per_device_batch, 'steps': steps, 'tokens': total_tokens, 'completion': completion}
Path('/kaggle/working/tpu-smoke-report.json').write_text(json.dumps(report, indent=2) + '\n')
print('TPU_V5E8_SMOKE_OK', flush=True)
